# 🚜 PAVI - Pavement AI Vision Inspector

**Vision Language Model for Road Defect Verification & Engineering Assessment**

This notebook uses **Qwen2.5-VL-7B-Instruct** to:
1. Verify automated pothole detections
2. Check for drainage issues
3. Identify landmarks for repair crews
4. Provide engineering recommendations
5. Share fun pavement facts!

---

## Setup Instructions
1. **GPU Required**: Use Colab with T4 GPU (Runtime → Change runtime type → T4 GPU)
2. **Runtime**: Python 3.10+
3. **Estimated Time**: ~5 min setup, ~10s per inference

## 📦 Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers accelerate torch torchvision pillow opencv-python moviepy einops

## 🤖 Step 2: Load PAVI (Qwen2.5-VL Model)

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch
import json

print("🔧 Loading PAVI Model...")
print("⏳ This may take 2-3 minutes on first run...\n")

# Model Selection
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"  # 7B is best for Colab T4
# For faster inference (lower quality): "Qwen/Qwen2.5-VL-2B-Instruct"

# Load Model
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)

print("✅ PAVI is ready!\n")
print(f"📊 Model: {model_id}")
print(f"🎯 Device: {model.device}")
print(f"💾 Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 🧠 Step 3: Define PAVI's Engineering Prompt

In [ ]:
PAVI_PROMPT = """You are **PAVI** (Pavement AI Vision Inspector), a Senior Civil Engineer who is not only a technical expert but also a friendly pavement guide.

Your expertise:
- Road and bridge paving techniques
- Defect diagnosis and rehabilitation strategies
- Explaining complex engineering concepts in simple terms

You love sharing "pavement facts" and helping people become "street smart" about infrastructure!

---

The automated system detected the following:
- Defects: {defect_count} potholes
- Density: {density}% (Yellow Zone - requires verification)
- Location: {location}

Analyze this road image and provide your expert assessment:

1. **VERIFY:** Do you confirm the presence of potholes? (Yes/No)
2. **DRAINAGE CHECK:** Is there evidence of poor drainage (standing water, clogged gutters, damaged shoulders)?
3. **LANDMARKS:** What landmarks can help a repair crew locate this section? (e.g., "50m past Petron gas station", "near Barangay Hall")
4. **FINAL RECOMMENDATION:** Based on your analysis, do you recommend (a) Patching Only, (b) Patching + Drainage Fix, or (c) Major Rehabilitation?
5. **PAVI FUN FACT:** Share one interesting fact about road maintenance or pavement engineering!

Respond in JSON format:
{{
  "verified": true/false,
  "drainage_issue": true/false,
  "landmarks": "description",
  "recommendation": "a|b|c",
  "reasoning": "brief explanation",
  "pavi_fun_fact": "your interesting pavement fact"
}}"""

print("✅ PAVI Prompt loaded!")

## 🎯 Step 4: Inference Function

In [ ]:
def pavi_analyze(image_path, defect_count=5, density=15.0, location="Segment 4"):
    """
    Run PAVI analysis on a road image.
    
    Args:
        image_path: Path to image file or PIL Image object
        defect_count: Number of potholes detected
        density: Pothole density percentage
        location: Road segment identifier
    
    Returns:
        dict: PAVI's structured assessment
    """
    # Load image
    if isinstance(image_path, str):
        image = Image.open(image_path).convert("RGB")
    else:
        image = image_path
    
    # Format prompt with context
    prompt = PAVI_PROMPT.format(
        defect_count=defect_count,
        density=density,
        location=location
    )
    
    # Prepare messages
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]}
    ]
    
    # Apply chat template
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True).to(model.device)
    
    # Generate response
    print("🤔 PAVI is analyzing...")
    output_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.7)
    response = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    
    # Parse JSON
    try:
        # Extract JSON block
        json_start = response.find("{")
        json_end = response.rfind("}") + 1
        if json_start != -1 and json_end > json_start:
            result = json.loads(response[json_start:json_end])
        else:
            # Fallback if no JSON found
            result = {"error": "No JSON found", "raw_response": response}
    except Exception as e:
        result = {"error": str(e), "raw_response": response}
    
    return result

print("✅ Inference function ready!")

## 🧪 Step 5: Test PAVI (Sample Image)

**Upload a road image** to test PAVI. You can:
1. Click the 📁 Files icon on the left sidebar
2. Upload a pothole image
3. Update the `image_path` below

In [ ]:
# Option 1: Use a sample image from the internet
from io import BytesIO
import requests

# Sample pothole image (public domain)
sample_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3e/Pothole_in_asphalt_road.jpg/800px-Pothole_in_asphalt_road.jpg"
response = requests.get(sample_url)
sample_image = Image.open(BytesIO(response.content))

# Display image
print("📸 Sample Image:")
display(sample_image.resize((400, 300)))

# Run PAVI Analysis
result = pavi_analyze(
    image_path=sample_image,
    defect_count=3,
    density=18.5,
    location="Sample Road Segment"
)

# Print Results
print("\n" + "="*60)
print("🚜 PAVI's Assessment")
print("="*60)
print(json.dumps(result, indent=2))

# Pretty print for readability
if "error" not in result:
    print("\n" + "-"*60)
    print(f"✅ Verified: {result.get('verified', 'N/A')}")
    print(f"💧 Drainage Issue: {result.get('drainage_issue', 'N/A')}")
    print(f"📍 Landmarks: {result.get('landmarks', 'N/A')}")
    print(f"🛠️ Recommendation: {result.get('recommendation', 'N/A')}")
    print(f"💡 PAVI Fun Fact: {result.get('pavi_fun_fact', 'N/A')}")
    print("-"*60)

## 🎬 Step 6: Batch Processing (Multiple Frames)

Use this to process multiple road segments efficiently.

In [ ]:
import os
from tqdm import tqdm

def batch_analyze(image_folder, output_json="pavi_results.json"):
    """
    Analyze all images in a folder.
    """
    results = []
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    for img_file in tqdm(image_files, desc="PAVI Processing"):
        img_path = os.path.join(image_folder, img_file)
        
        try:
            result = pavi_analyze(img_path)
            results.append({
                "image": img_file,
                "assessment": result
            })
        except Exception as e:
            print(f"❌ Failed on {img_file}: {e}")
    
    # Save results
    with open(output_json, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✅ Batch complete! Results saved to {output_json}")
    return results

# Example usage (uncomment to run):
# results = batch_analyze("path/to/your/images")

## 📊 Step 7: Export for Express AI Backend

Convert PAVI's output to the format expected by the Express AI API.

In [ ]:
def export_for_backend(pavi_result, segment_id):
    """
    Format PAVI output for Express AI backend.
    """
    return {
        "segment_id": segment_id,
        "vlm_verified": pavi_result.get("verified", False),
        "drainage_issue_detected": pavi_result.get("drainage_issue", False),
        "landmarks": pavi_result.get("landmarks", ""),
        "vlm_recommendation": pavi_result.get("recommendation", "a"),
        "vlm_reasoning": pavi_result.get("reasoning", ""),
        "pavi_insight": pavi_result.get("pavi_fun_fact", "")
    }

# Example
if "error" not in result:
    backend_payload = export_for_backend(result, segment_id="SEG_001")
    print("📤 Backend Payload:")
    print(json.dumps(backend_payload, indent=2))

## 🚀 Next Steps

1. **Integrate with Backend**: Create `/api/v1/vlm/analyze` endpoint to accept this data
2. **Live Dashboard**: Display PAVI's insights in the `DiagnosisCard` component
3. **Production Deployment**: Move to local GPU or cloud (RunPod/Lambda Labs)

---

## 📝 Notes

- **Cost**: FREE on Colab (T4 GPU)
- **Speed**: ~10-15 seconds per image on T4
- **Privacy**: Data stays in your Colab session (no external API calls)
- **Model Size**: ~14GB (7B model)

**Created by**: Express AI Development Team  
**Model**: Alibaba Qwen2.5-VL-7B-Instruct  
**License**: Apache 2.0